# Step by Step Guide to use this GraphSage Implementation

This notebook provides a step-by-step guide to using the GraphSage implementation provided in this repository. It covers the necessary imports, model setup, and training process. One can follow along to understand how to utilize the implemented GraphSage algorithm for node classification tasks. If one just wants to run the code, they can run python main.py with the appropriate arguments, but this notebook is meant to provide a deeper understanding of the code and its components.

Before starting, make sure to have the necessary libraries installed and the dataset ready. The code is structured to be modular, allowing for easy modifications and experimentation with different configurations. One can install the required libraries using pip install -r requirements.txt and ensure that the dataset is in the correct format and location as expected by the code.

## Setup 

In [12]:
import sys
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import torch.nn.functional as F
from src.Algo_Mini_Batch import AlgoMiniBatch
from src.train import train
from src.layers import MeanAggregator, MaxPoolingAggregator, LSTMAggregator

In [ ]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### Choice and loading of the graph dataset

Three graphs are avaible by default: ppi, reddit and cora. You can also generate your own graph dataset using the script in src/generation_dataset/generate_graph_dataset.py and following the instructions in the README.md file.

In [ ]:
# Choix du graphe : "cora", "ppi", "pubmed"
GRAPH_NAME = "cora"
GRAPH_PATH = f"src/generation_dataset/2026-02-10_14-21-11/graphs/{GRAPH_NAME}.pt"

G = torch.load(GRAPH_PATH, weights_only=False)

print("Graph loaded:", GRAPH_NAME)
print("Nb nodes :", G.number_of_nodes())
print("Nb edges :", G.number_of_edges())

node0 = list(G.nodes())[0]
assert "x" in G.nodes[node0], "ERREUR : pas de features 'x'"
feat_dim = G.nodes[node0]["x"].shape[0]
print(f"Feature dim : {feat_dim}")

# Cast en float32
for node in G.nodes():
    G.nodes[node]["features"] = G.nodes[node]["x"].float()

# Normalisation z-score des features
# Sans ça, si les features sont toutes négatives, ReLU tue le signal
all_feats = torch.stack([G.nodes[n]["features"] for n in G.nodes()])
feat_mean = all_feats.mean(dim=0)
feat_std  = all_feats.std(dim=0).clamp(min=1e-6)
for node in G.nodes():
    G.nodes[node]["features"] = (G.nodes[node]["features"] - feat_mean) / feat_std

check = torch.stack([G.nodes[n]["features"] for n in G.nodes()])
print(f"\nFeatures normalisées — moyenne: {check.mean().item():.4f}  std: {check.std().item():.4f}")

### Aggregator choice

Three aggregators are implemented and can be used: MeanAggregator, MaxPoolingAggregator and LSTMAggregator. 

In [ ]:
# Choix de l'agrégateur : "mean", "max", "lstm"
AGGREGATOR = "mean"

# Choix du nombre de couches K (papier : K=2)
depth = 2

# hidden_dim fixé à 256 comme dans le papier (indépendant de feat_dim)
hidden_dim = 256

if AGGREGATOR == "mean":
    agg = nn.ModuleList([MeanAggregator() for _ in range(depth)])
elif AGGREGATOR == "max":
    agg = nn.ModuleList(
        [MaxPoolingAggregator(in_features=feat_dim, out_features=hidden_dim)] +
        [MaxPoolingAggregator(in_features=hidden_dim, out_features=hidden_dim) for _ in range(depth - 1)]
    )
elif AGGREGATOR == "lstm":
    agg = nn.ModuleList(
        [LSTMAggregator(in_features=feat_dim, out_features=hidden_dim)] +
        [LSTMAggregator(in_features=hidden_dim, out_features=hidden_dim) for _ in range(depth - 1)]
    )
else:
    raise ValueError(f"Agrégateur inconnu : {AGGREGATOR}. Choisir parmi 'mean', 'max', 'lstm'.")

print(f"Agrégateur : {AGGREGATOR}  |  Profondeur : {depth}")
print(f"feat_dim={feat_dim}  →  hidden_dim={hidden_dim}")

## Model and training

In [ ]:
# Nombre de voisins échantillonnés par couche (S1, S2 dans le papier)
SAMPLE_SIZE = 10

# W[0] : feat_dim + agg_output_dim → hidden_dim
# W[1:] : 2*hidden_dim → hidden_dim
# Pour MeanAggregator : agg_output_dim = feat_dim (préserve la dim)
# Pour Max/LSTM       : agg_output_dim = hidden_dim (projection interne)
agg_out_0 = agg[0].output_dim(feat_dim)

W = nn.ModuleList(
    [nn.Linear(feat_dim + agg_out_0, hidden_dim)] +
    [nn.Linear(2 * hidden_dim, hidden_dim) for _ in range(depth - 1)]
)

# Initialisation Kaiming adaptée à LeakyReLU
for layer in W:
    nn.init.kaiming_uniform_(layer.weight, nonlinearity='leaky_relu', a=0.1)
    nn.init.zeros_(layer.bias)

sample_size = [SAMPLE_SIZE for _ in range(depth)]

activation = lambda x: F.leaky_relu(x, negative_slope=0.1)

model = AlgoMiniBatch(depth, W, activation, agg,
                      normalize_output=True, in_features=feat_dim).to(device)

print(f"W[0] : {W[0].in_features} → {W[0].out_features}")
for i in range(1, depth):
    print(f"W[{i}] : {W[i].in_features} → {W[i].out_features}")
print(f"Nb paramètres : {sum(p.numel() for p in model.parameters()):,}")
print(f"Device : {next(model.parameters()).device}")

In [ ]:
EPOCHS     = 10
BATCH_SIZE = 128
LR         = 1e-3
NUM_PAIRS  = 50000  # paires positives par random walks (papier : ~677k pour Cora)

print("--------------------------------")
print("La phase d'entrainement débute..")

train(
    model,
    G,
    device,
    sampling_size=sample_size,
    epochs=EPOCHS,
    learning_rate=LR,
    batch_size=BATCH_SIZE,
    num_pairs=NUM_PAIRS,
)

In [ ]:
model.eval()

all_nodes = list(G.nodes())

with torch.no_grad():
    embeddings = model.forward_propagation(G, all_nodes, sample_size)

print(f"Shape embeddings : {embeddings.shape}")
print(f"Min  : {embeddings.min().item():.4f}")
print(f"Max  : {embeddings.max().item():.4f}")
print(f"Mean : {embeddings.mean().item():.4f}")
print(f"Std  : {embeddings.std().item():.4f}")

norms = embeddings.norm(dim=1)
print(f"\nNormes L2 — min: {norms.min():.4f}  max: {norms.max():.4f}  mean: {norms.mean():.4f}")

# Similarité cosine moyenne entre les 5 premiers nœuds (embeddings déjà normalisés)
sim_matrix = embeddings[:5] @ embeddings[:5].T
print(f"\nMatrice de similarité cosine (5 premiers nœuds) :")
print(sim_matrix.cpu().numpy().round(3))

## Benchmark : graphs × aggregators

Entraîne chaque combinaison (graph, agrégateur) et affiche les courbes de loss.

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F_bench

# ── Hyperparamètres du benchmark ──────────────────────────────────────────────
BENCH_GRAPHS      = ["cora", "ppi", "pubmed"]
BENCH_AGGREGATORS = ["mean", "max", "lstm"]
BENCH_EPOCHS      = 10
BENCH_BATCH_SIZE  = 128
BENCH_LR          = 1e-3
BENCH_NUM_PAIRS   = 20000   # réduit pour aller vite ; augmenter pour plus de précision
BENCH_SAMPLE_SIZE = 10
BENCH_DEPTH       = 2
BENCH_HIDDEN_DIM  = 256
GRAPH_BASE        = "src/generation_dataset/2026-02-10_14-21-11/graphs"

# ── Helpers ───────────────────────────────────────────────────────────────────

def bench_load_graph(name):
    import torch
    G = torch.load(f"{GRAPH_BASE}/{name}.pt", weights_only=False)
    node0 = list(G.nodes())[0]
    feat_dim = G.nodes[node0]["x"].shape[0]
    for node in G.nodes():
        G.nodes[node]["features"] = G.nodes[node]["x"].float()
    all_feats = torch.stack([G.nodes[n]["features"] for n in G.nodes()])
    mu  = all_feats.mean(dim=0)
    std = all_feats.std(dim=0).clamp(min=1e-6)
    for node in G.nodes():
        G.nodes[node]["features"] = (G.nodes[node]["features"] - mu) / std
    return G, feat_dim

def bench_build_model(feat_dim, agg_type, depth=BENCH_DEPTH, hidden_dim=BENCH_HIDDEN_DIM):
    import torch.nn as nn
    import torch.nn.functional as F
    if agg_type == "mean":
        agg = nn.ModuleList([MeanAggregator() for _ in range(depth)])
    elif agg_type == "max":
        agg = nn.ModuleList(
            [MaxPoolingAggregator(feat_dim, hidden_dim)] +
            [MaxPoolingAggregator(hidden_dim, hidden_dim) for _ in range(depth - 1)]
        )
    elif agg_type == "lstm":
        agg = nn.ModuleList(
            [LSTMAggregator(feat_dim, hidden_dim)] +
            [LSTMAggregator(hidden_dim, hidden_dim) for _ in range(depth - 1)]
        )
    agg_out_0 = agg[0].output_dim(feat_dim)
    W = nn.ModuleList(
        [nn.Linear(feat_dim + agg_out_0, hidden_dim)] +
        [nn.Linear(2 * hidden_dim, hidden_dim) for _ in range(depth - 1)]
    )
    for layer in W:
        nn.init.kaiming_uniform_(layer.weight, nonlinearity='leaky_relu', a=0.1)
        nn.init.zeros_(layer.bias)
    activation = lambda x: F.leaky_relu(x, negative_slope=0.1)
    return AlgoMiniBatch(depth, W, activation, agg, normalize_output=True, in_features=feat_dim)

# ── Entraînement de toutes les combinaisons ───────────────────────────────────

results = {}   # (graph_name, agg_type) -> List[float]
sample_sizes = [BENCH_SAMPLE_SIZE] * BENCH_DEPTH

for graph_name in BENCH_GRAPHS:
    print(f"\n{'='*60}")
    print(f"  Graphe : {graph_name}")
    print(f"{'='*60}")
    G_bench, feat_dim_bench = bench_load_graph(graph_name)
    print(f"  {G_bench.number_of_nodes()} nœuds  |  {G_bench.number_of_edges()} arêtes  |  feat_dim={feat_dim_bench}")

    for agg_type in BENCH_AGGREGATORS:
        print(f"\n  Agrégateur : {agg_type}")
        model_bench = bench_build_model(feat_dim_bench, agg_type).to(device)
        losses = train(
            model_bench, G_bench, device,
            sampling_size=sample_sizes,
            epochs=BENCH_EPOCHS,
            learning_rate=BENCH_LR,
            batch_size=BENCH_BATCH_SIZE,
            num_pairs=BENCH_NUM_PAIRS,
        )
        results[(graph_name, agg_type)] = losses

# ── Tracé des courbes ──────────────────────────────────────────────────────────

BASELINE = 14.5561   # -log(σ(0)) × (1 + 20),  Q=20
COLORS   = {"mean": "#1f77b4", "max": "#ff7f0e", "lstm": "#2ca02c"}
MARKERS  = {"mean": "o", "max": "s", "lstm": "^"}

fig, axes = plt.subplots(1, len(BENCH_GRAPHS), figsize=(5 * len(BENCH_GRAPHS), 4), sharey=False)
if len(BENCH_GRAPHS) == 1:
    axes = [axes]

epochs_x = list(range(1, BENCH_EPOCHS + 1))

for ax, graph_name in zip(axes, BENCH_GRAPHS):
    ax.axhline(BASELINE, color="gray", linestyle="--", linewidth=1, label="baseline (aléatoire)")
    for agg_type in BENCH_AGGREGATORS:
        losses = results[(graph_name, agg_type)]
        ax.plot(epochs_x, losses,
                color=COLORS[agg_type], marker=MARKERS[agg_type],
                markersize=4, linewidth=1.5, label=agg_type)
    ax.set_title(graph_name, fontsize=13, fontweight="bold")
    ax.set_xlabel("Époque")
    ax.set_ylabel("Loss moyenne")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle("Courbes d'entraînement GraphSAGE", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
